# <center><font size = 5><span style="color:#287c2d;font-family:'Cambria'">It is important that credit card companies are able to recognize fraudulent credit card transactions so that customers are not charged for items that they did not purchase </span></font></center>

# <center><font size =5><span style="font-family:'Sans';color:#2B13DC"> Please don't forget to Upvote 👆 if you find it useful :)👍👍 </span></font></center>


# <center><font size = 4><span style="color:#080743"> <p style="background-color:#080743;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:50px 10px;">About the dataset </p>   </span></font></center>

* The dataset contains transactions made by credit cards in September 2013 by European cardholders.

* This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions.

* The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.

* It contains only numerical input variables which are the result of a PCA transformation. 

* Features V1, V2, … V28 are the principal components obtained with PCA 

* The only features which have not been transformed with PCA are `Time` and `Amount`.

* Feature `Time` contains the seconds elapsed between each transaction and the first transaction in the dataset. 

* The feature `Amount` is the transaction Amount, this feature can be used for example-dependant cost-sensitive learning.

* Feature `Class` is the response variable and it takes value 1 in case of fraud and 0 otherwise.

<div style = 'border: 3px solid #D9C10B;'>

<p style="background-color:#B81868;font-family:serif;color:#f1faee;font-size:150%;text-align:center;border-radius:100px 8px;">Importing the required libraries</p>

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense,Dropout, BatchNormalization
from keras import regularizers
import numpy as np 
import pandas as pd 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

<p style="background-color:#B81868;font-family:serif;color:#f1faee;font-size:150%;text-align:center;border-radius:100px 8px;"> Reading the original dataset (as dataframe) available on this link: </p>

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

In [ ]:
original_df = pd.read_csv('/kaggle/input/creditcardfraud/creditcard.csv')
original_df.info()

<p style="background-color:#B81868;font-family:serif;color:#f1faee;font-size:150%;text-align:center;border-radius:10px;">  The dataset provided for the competition is imported in `train_df` and `test_df` dataframes </p>

In [ ]:
train_df = pd.read_csv('/kaggle/input/playground-series-s3e4/train.csv')
train_df = pd.concat([train_df,original_df])
train_df = train_df.sample(frac=1)
train_df = train_df.reset_index(drop=True)
y_train = train_df['Class']
y_train_original = original_df['Class']
train_df.head()

In [ ]:
y_train.value_counts()

## We can observe here that the dataset is highly unbalanced. 
###  My strategy would be to upsample the dataset provided and then merge the original dataset. 
###  The original Dataset won't be upsampled to counter overfitting cases. 
###  Also ,the objective would be to maximise AUC & Recall and Precision to minimise the imbalance while training
[VERSION 12 UPDATE] : As of now, I'm using PrecisonAtRecall : This metric tries to increase the precision for specified recall(here 0.95) .
                      [https://www.tensorflow.org/api_docs/python/tf/keras/metrics/PrecisionAtRecall] 

In [ ]:
test_df = pd.read_csv('/kaggle/input/playground-series-s3e4/test.csv')
test_df_id = test_df['id']
test_df.head()

In [ ]:
train_df.columns

In [ ]:
train_df.describe()

 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;">Storing the numerical columns for scaling and upsampling the respective values </p>   </span></font></center>

In [ ]:
train_to_scale = train_df[['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
       'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
       'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']]

test_to_scale = test_df[['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
       'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
       'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']]



## Now, we will be doing SMOTE (Synthetic Minority Oversampling Technique) Upsampling 
#### SMOTE is applied to create new synthetic minority samples to get a balanced distribution. 

In [ ]:
from imblearn.over_sampling import SMOTE
smt = SMOTE(random_state=0,n_jobs = -1)
train_to_scale, y_train = smt.fit_resample(train_to_scale, y_train)


## Let's check the distribution across the columns of the dataset

### Things to keep in mind for better interpretation of plot:
#### 1. Kurtosis : It is a statistical measure of whether the data is heavy-tailed or light-tailed in a normal distribution
#### Normal Distribution has kurtosis of 3. Hence Excess Kurtuosis is calculate as Kurt - 3, where kurt is the actual kurtusis of distribution
#### The higher the excess kurtosis from 0, the higher the noise in the model and at the end the higher the misclassification rate
Ref : https://www.matec-conferences.org/articles/matecconf/pdf/2018/63/matecconf_imiec2018_02018.pdf

In [ ]:
fig, axs = plt.subplots(3, 10, figsize=(49, 30))
fig.subplots_adjust(wspace=0.5)
axs = axs.ravel()
cmap = sns.color_palette("RdBu", as_cmap=True)
skewness = train_to_scale.skew()
kurtosis = train_to_scale.kurtosis()
for i, column in enumerate(train_to_scale.columns):
    axs[i].set_title(column)
    axs[i].hist(train_to_scale[column], bins=200, alpha=0.5,color  = plt.cm.RdYlGn(1))
    axs[i].set_xlabel(column)
    axs[i].set_ylabel('Probability')
    axs[i].annotate(text='Skewness: {:.2f}\nKurtosis: {:.2f}'.format(skewness[column], kurtosis[column]), xy=(0.2, 0.9), xycoords='axes fraction', fontsize=12)
plt.show()


### Checking correlation before applying transformation

In [ ]:
fig, ax = plt.subplots(figsize=(30,20))  
sns.heatmap(train_to_scale.corr(),annot=True,cmap = "crest")
plt.show()

### V2,V5,V6,V7,V8,V10,V12,V14,V17,V20,V21,V23,V27,V28 and Amount are highly skewed and Leptokurtic(Excessive Kurtusis >3) as well . Applying yeojhonson and log transformation on all of them
#### Link : https://www.stat.umn.edu/arc/yjpower.pdf

In [ ]:
from sklearn.preprocessing import power_transform
#transformed_data = (original_data, method='box-cox')

column = [ 'V2','V5', 'V6', 'V7', 'V8','V10','V17','V12','V14','V20', 'V21','V23','V27', 'V28','Amount']
for  i in column : 
    train_to_scale[[i]]= (power_transform(train_to_scale[[i]],method = 'yeo-johnson'))



In [ ]:
fig, axs = plt.subplots(3, 10, figsize=(49, 20))
fig.subplots_adjust(wspace=0.5)
axs = axs.ravel()
cm = plt.cm.get_cmap('RdYlBu_r')
skewness = train_to_scale.skew()
kurtosis = train_to_scale.kurtosis()
for i, column in enumerate(train_to_scale.columns):
    
    axs[i].set_title(column)
    n, bins, patches  = axs[i].hist(train_to_scale[column], bins=200, alpha=0.9)
    axs[i].annotate(text='Skewness: {:.2f}\nKurtosis: {:.2f}'.format(skewness[column], kurtosis[column]), xy=(0.2, 0.9), xycoords='axes fraction', fontsize=12)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # scale values to interval [0,1]
    col = bin_centers - min(bin_centers)
    col /= max(col)

    for c, p in zip(col, patches):
        plt.setp(p, 'facecolor', cm(c))
        axs[i].set_xlabel(column)
        axs[i].set_ylabel('Probability')
    
plt.show()


### Let's scale the data to check the unwanted high feature importance for a particular feature

In [ ]:
scaler = RobustScaler()

scaled_train = pd.DataFrame(scaler.fit_transform(train_to_scale),columns = train_to_scale.columns)
scaled_test = pd.DataFrame(scaler.transform(test_to_scale),columns = test_to_scale.columns)
X_train = scaled_train
X_test = scaled_test


 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;">Defining the Model Architecture and appropriate loss function and metrices </p>   </span></font></center>

In [ ]:
model = Sequential()

# Add layers to the model
model.add(Dense(8116, input_dim=30, activation='relu')) #input layer with 64 neurons
model.add(Dropout(0.5))
model.add(Dense(2048,activation= 'relu'))
model.add(Dropout(0.5))
model.add(Dense(1024,activation= 'relu'))
model.add(Dropout(0.2))
model.add(Dense(256,activation = 'relu'))
model.add(Dropout(0.2))
model.add(Dense(128,activation = 'relu'))
model.add(Dense(8,activation= 'relu'))
model.add(Dense(1, activation='sigmoid')) #output layer with 1 neuron 
model.compile(loss=['binary_crossentropy'], optimizer= tf.keras.optimizers.Nadam(4e-6),metrics = [tf.keras.metrics.AUC(curve='ROC')])
model.summary()

 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;">Training the model with validation_split = 0.26 </p>   </span></font></center>

In [ ]:
model.fit(X_train,y_train,validation_split = 0.26,batch_size = 4096,epochs = 25)#,callbacks=[callbacks])

 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;"> Getting the feature importances </p>   </span></font></center>

In [ ]:
# Get the weights of the first layer
weights = model.layers[0].get_weights()[0]

# Get the absolute values of the weights
importances = np.abs(weights)

# Normalize the importances
importances = importances / importances.sum(axis=0)

# Print the importances of each feature
for i, importance in enumerate(importances):
    print("Feature", i, "Importance", np.median(importance))


 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;">Prediction on X_test </p>   </span></font></center> 

In [ ]:
y_pred = model.predict(X_test)
print(y_pred)

In [ ]:
y_pred_train = model.predict(X_train)
print(y_pred_train)

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Assume y_true and y_scores are the true labels and predicted scores for your dataset
precision, recall, thresholds = precision_recall_curve(y_train, y_pred_train)

# Plot the precision-recall curve
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

 # <center><font size = 2><span style="color:#F5F5E6"> <p style="background-color:#C76E09;font-family:newtimeroman;color:#FFFFFF;font-size:300%;text-align:center;border-radius:10px 10px;">Conversion of DataFrame in required format for submission to the competition </p>   </span></font></center> 

In [ ]:
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ['Class']
submissions_df = pd.DataFrame(pd.concat([test_df_id,y_pred],axis = 1))
submissions_df = submissions_df.reset_index(drop = True)

submissions_df.to_csv('submission.csv', index=False)
submissions_df.head()

### Please upvote if you find it useful. Thanks